In [1]:
# Cell 1: Imports and Directory Management
import sys
import os
import glob
import math
import pandas as pd
import pypowsybl as pp
import pypowsybl.network as pn
from unified_generator import generate_dynamic_files


NOTEBOOK_NAME = "OM_test_1"
OUTPUT_DIR = os.path.join(os.getcwd(), f"{NOTEBOOK_NAME}_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Environment ready. Outputs will be saved in: {OUTPUT_DIR}")

# Setup for OpenModelica (Uncomment if OMPython is installed)
# from OMPython import OMCSessionZMQ
# omc = OMCSessionZMQ()

Environment ready. Outputs will be saved in: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_1_outputs


In [2]:
# Cell 2: Define the Network Creation Logic


def create_example_network(nom_v=1.0, sn_ref=100.0):
    """
    Creates a simple network with a TSO slack bus and a Generator.
    """
    zb = nom_v * nom_v / sn_ref

    network = pp.network.create_empty()

    # 1. Create Substations
    network.create_substations(
        id=["TSONetwork", "Producer"],
        name=["TSONetwork", "Producer"],
        country=["FR", "FR"],
        tso=["TSO", "TSO"],
    )

    # 2. Create Voltage Levels
    network.create_voltage_levels(
        id="TSONetwork_VL",
        name="TSONetwork_VL",
        substation_id="TSONetwork",
        topology_kind="BUS_BREAKER",
        nominal_v=nom_v,
    )
    network.create_voltage_levels(
        id="Producer_VL",
        name="Producer_VL",
        substation_id="Producer",
        topology_kind="BUS_BREAKER",
        nominal_v=nom_v,
    )

    # 3. Create Buses
    network.create_buses(id="TSONetwork_Bus", voltage_level_id="TSONetwork_VL")
    network.create_buses(id="Producer_Bus", voltage_level_id="Producer_VL")

    # 4. Create Lines
    network.create_lines(
        id="Line_TSO_Prod",
        voltage_level1_id="TSONetwork_VL",
        bus1_id="TSONetwork_Bus",
        voltage_level2_id="Producer_VL",
        bus2_id="Producer_Bus",
        x=0.1 * zb,
        r=0.01 * zb,
    )

    # 5. Create Generators

    # --- TSO Infinite Bus (Slack) ---
    network.create_generators(
        id="InfiniteBus",
        voltage_level_id="TSONetwork_VL",
        bus_id="TSONetwork_Bus",
        target_p=0.0,
        target_v=1.0 * nom_v,  # Slack maintains Voltage
        voltage_regulator_on=True,
        min_p=-10000.0,
        max_p=10000.0,
    )

    # --- The Dynamic Generator (GenUnit) ---
    network.create_generators(
        id="GenUnit",
        voltage_level_id="Producer_VL",
        bus_id="Producer_Bus",
        target_p=50.0,  # Active Power setpoint
        target_v=1.0 * nom_v,  # <--- NEW: Voltage Setpoint (Required if regulator is ON)
        energy_source="THERMAL",
        rated_s=100.0,
        min_p=0.0,
        max_p=90.0,
        voltage_regulator_on=True,
    )

    return network


# Instantiate the network
network = create_example_network()
print("Network created successfully.")

Network created successfully.


In [3]:
# Cell 3: Run Static Power Flow (Initialization Step 1)
import os

# Define LoadFlow parameters
parameters = pp.loadflow.Parameters(
    distributed_slack=False,
    provider_parameters={"slackBusSelectionMode": "NAME", "slackBusesIds": "InfiniteBus"},
)

print("Running AC Loadflow...")
results = pp.loadflow.run_ac(network, parameters=parameters)

# Check status
status_name = (
    results[0].status.name if hasattr(results[0].status, "name") else str(results[0].status)
)

if status_name == "CONVERGED":
    print(f"Power Flow Status: {status_name}")
else:
    print(f"WARNING: Power Flow did not converge. Status: {status_name}")

# Display results
gen_results = network.get_generators(all_attributes=True)
print("\nGenerator Status after LoadFlow:")
print(gen_results[["p", "q", "target_p", "target_v"]])

# Using .xiidm is the standard for the XIIDM format in newer versions
iidm_filename = os.path.join(OUTPUT_DIR, "initialized_network.xiidm")

network.save(iidm_filename, format="XIIDM", parameters={"iidm.export.xml.version": "1.4"})

# Verify the file exists immediately
if os.path.exists(iidm_filename):
    print(f"Success: File saved as '{iidm_filename}'")
else:
    print(f"Error: File '{iidm_filename}' was not created. Check permissions.")

Running AC Loadflow...
Power Flow Status: CONVERGED

Generator Status after LoadFlow:
                p    q  target_p  target_v
id                                        
InfiniteBus  -0.0 -0.0       0.0       1.0
GenUnit     -50.0 -0.0      50.0       1.0
Success: File saved as '/home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_1_outputs/initialized_network.xiidm'


In [4]:
# Cell 4: Generate Dynamic Configuration (Dynawo Files)

# The filename must match what we saved in Cell 3
iidm_filename = os.path.join(OUTPUT_DIR, "initialized_network.xiidm")

if os.path.exists(iidm_filename):
    generate_dynamic_files(iidm_filename, simulator="dynawo")
    print("Dynamic files (.dyd and .par) generated successfully.")
else:
    print(f"Error: Could not find '{iidm_filename}'. Please run Cell 3 again.")

--- Generating configuration for DYNAWO using /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_1_outputs/initialized_network.xiidm ---
Successfully generated files for dynawo.
Dynamic files (.dyd and .par) generated successfully.


In [5]:
# Cell 5: Automated Initialization for OMJulia/OMPython
# This step extracts the specific boundary conditions needed for Modelica initialization.


def get_initialization_values(network, generator_id):
    """
    Extracts P, Q, V, and Theta from the PowSyBl network for a specific generator.
    Returns a dictionary suitable for injecting into a Modelica initialization script.
    """
    # Get generators and buses dataframes
    gens = network.get_generators(all_attributes=True)
    buses = network.get_buses(all_attributes=True)

    if generator_id not in gens.index:
        raise ValueError(f"Generator {generator_id} not found.")

    # Get P and Q (Note: PowSyBl P is usually generation > 0, load < 0)
    p_init = gens.at[generator_id, "p"]
    q_init = gens.at[generator_id, "q"]

    # Get Voltage magnitude and Angle at the connecting bus
    bus_id = gens.at[generator_id, "bus_id"]
    v_mag = buses.at[bus_id, "v_mag"]
    v_angle_deg = buses.at[bus_id, "v_angle"]

    # Nominal voltage for PU calculation
    vl_id = gens.at[generator_id, "voltage_level_id"]
    # Getting nominal V is slightly complex in generic pandas, assuming 1.0 here or fetching from VL
    # In a full script we would fetch the VL table.

    return {
        "P_gen_init": p_init,
        "Q_gen_init": q_init,
        "V_mag_pu": v_mag,  # Assuming v_mag is in PU if base was handled, otherwise divide by Un
        "Angle_deg": v_angle_deg,
    }


# Get values for our specific unit
init_vals = get_initialization_values(network, "GenUnit")

print("--- Initialization Values for Modelica ---")
for key, val in init_vals.items():
    print(f"{key}: {val:.4f}")

print("\n--- Example OMJulia Command Generation ---")
# This creates a string you would pass to omc.sendExpression()
# Assuming the Modelica model has top-level parameters matching these names
om_commands = [
    f"parameter Real P0 = {init_vals['P_gen_init']};",
    f"parameter Real Q0 = {init_vals['Q_gen_init']};",
    f"parameter Real V0 = {init_vals['V_mag_pu']};",
    f"parameter Real A0 = {init_vals['Angle_deg']};",
    "simulate(YourModelName, stopTime=1.0)",
]

print("Commands ready to be sent to OMJulia:")
print("\n".join(om_commands))

--- Initialization Values for Modelica ---
P_gen_init: -50.0000
Q_gen_init: -0.0000
V_mag_pu: 1.0000
Angle_deg: 0.0000

--- Example OMJulia Command Generation ---
Commands ready to be sent to OMJulia:
parameter Real P0 = -50.0;
parameter Real Q0 = -0.0;
parameter Real V0 = 1.0;
parameter Real A0 = 0.0;
simulate(YourModelName, stopTime=1.0)
